# Obligatorio 2 — Problema 1

Entrenamiento y comparación de modelos de regresión sobre datos sintéticos, y análisis del comportamiento de las predicciones cuando el conjunto de test sufre un **desplazamiento de la media** (*shift*). El PCA se usa como herramienta para detectar y visualizar ese desplazamiento.

Detalle del enunciado y de los pasos: [`../RESUME.md`](../RESUME.md).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.linear_model import BayesianRidge, LinearRegression, RidgeCV
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## 0. Configuración

Todas las decisiones globales (semilla, tamaños, variables informativas y coeficientes) se fijan acá una sola vez.
Cambiar cualquiera de estos valores obliga a rehacer el pipeline completo.

In [ ]:
SEED = 42
N = 150          # observaciones totales
P = 50           # variables
N_TRAIN = 100
N_TEST = 50
NOISE_VAR = 0.25 # varianza del ruido gaussiano (sigma = 0.5)

# Variables que realmente explican Y (las demás tienen coeficiente 0)
INFORMATIVAS = [3, 11, 24, 37, 45]
COEFICIENTES = [3.0, -2.0, 1.5, 2.5, -1.0]

print(f"N={N}  P={P}  train={N_TRAIN}  test={N_TEST}")
print("Variables informativas:", INFORMATIVAS)

## 1. Datos sintéticos (150 × 50)

Se carga la matriz de covarianzas provista en el EVA y se generan los datos con una normal multivariada.

In [ ]:
sigma = pd.read_csv("mat_cov.csv").to_numpy()
columnas = [f"V{j + 1}" for j in range(P)]

assert sigma.shape == (P, P), sigma.shape
assert np.allclose(sigma, sigma.T), "la matriz debe ser simétrica"

autovalores = np.linalg.eigvalsh(sigma)
print("autovalor mínimo:", autovalores.min())
assert autovalores.min() > 0, "la matriz debe ser definida positiva"

rng = np.random.default_rng(SEED)
X = rng.multivariate_normal(mean=np.zeros(P), cov=sigma, size=N)
print("X:", X.shape)

## 2. Variable objetivo

$Y = \sum_{j=1}^{50} \beta_j X_j + \varepsilon$, con solo **5 coeficientes no nulos** y
$\varepsilon \sim \mathcal{N}(0,\, 0.25)$.

In [ ]:
beta_true = np.zeros(P)
beta_true[INFORMATIVAS] = COEFICIENTES

ruido = rng.normal(loc=0.0, scale=np.sqrt(NOISE_VAR), size=N)
y = X @ beta_true + ruido

var_senal = np.var(X @ beta_true)
print(f"varianza de la señal: {var_senal:.2f}")
print(f"varianza del ruido : {NOISE_VAR:.2f}")
print(f"R² teórico aprox.  : {var_senal / (var_senal + NOISE_VAR):.3f}")

## 3. Separación train / test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=N_TRAIN, test_size=N_TEST, random_state=SEED
)
print("train:", X_train.shape, "| test:", X_test.shape)

## 4. Entrenamiento de los cuatro modelos

### 4.1 Escalado

Ridge y PCA no son invariantes a escala, así que se estandariza usando **solo** las medias y desvíos de train.

In [ ]:
scaler = StandardScaler().fit(X_train)
X_train_std = scaler.transform(X_train)
X_test_std = scaler.transform(X_test)

# beta expresado en la escala estandarizada (para comparar con los coeficientes estimados)
beta_true_std = beta_true * scaler.scale_

### 4.2 Regresión lineal simple (OLS)

In [ ]:
ols = LinearRegression().fit(X_train_std, y_train)
print(f"MSE train: {np.mean((y_train - ols.predict(X_train_std)) ** 2):.4f}")

### 4.3 Regresión Ridge

El hiperparámetro $\alpha$ se elige por validación cruzada.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

ridge = RidgeCV(alphas=np.logspace(-3, 3, 60), cv=cv).fit(X_train_std, y_train)
print(f"alpha elegido: {ridge.alpha_:.4g}")
print(f"MSE train   : {np.mean((y_train - ridge.predict(X_train_std)) ** 2):.4f}")

### 4.4 Regresión bayesiana

In [ ]:
bayes = BayesianRidge().fit(X_train_std, y_train)
print(f"alpha={bayes.alpha_:.4g}  lambda={bayes.lambda_:.4g}")
print(f"MSE train: {np.mean((y_train - bayes.predict(X_train_std)) ** 2):.4f}")

### 4.5 Regresión sobre componentes principales (PCR)

Se elige el número de componentes $k$ por validación cruzada. Luego se entrenan PCA y regresión lineal con ese $k$.

Los pesos de cada variable en el modelo se recuperan combinando los coeficientes sobre las componentes con los *loadings*:

$$\hat{\beta}_{\text{std}} = V_{:,1:k}\,\hat{\gamma}, \qquad \hat{\beta}_{\text{orig}} = \hat{\beta}_{\text{std}} \oslash s$$

donde $V$ son los autovectores (filas de `pca.components_`), $\hat{\gamma}$ los coeficientes de la regresión sobre los scores y $s$ los desvíos estándar usados al escalar.

In [ ]:
pcr_pipeline = make_pipeline(PCA(), LinearRegression())

busqueda_k = GridSearchCV(
    pcr_pipeline,
    param_grid={"pca__n_components": range(1, P + 1)},
    cv=cv,
    scoring="neg_mean_squared_error",
)
busqueda_k.fit(X_train_std, y_train)

k_optimo = busqueda_k.best_params_["pca__n_components"]
pcr = busqueda_k.best_estimator_
print(f"k elegido por CV: {k_optimo}")
print(f"MSE CV          : {-busqueda_k.best_score_:.4f}")


def coefs_pcr_en_std(pipeline):
    """Pasa los coeficientes del PCR a la escala de las 50 variables estandarizadas."""
    pca = pipeline.named_steps["pca"]
    reg = pipeline.named_steps["linearregression"]
    return pca.components_[: pca.n_components_].T @ reg.coef_


beta_pcr_std = coefs_pcr_en_std(pcr)
print("coeficientes recuperados:", beta_pcr_std.shape)

### 4.6 Comparación de coeficientes

Se grafican los 50 pesos estimados por cada modelo contra los verdaderos.
En rojo, las 5 variables que realmente contienen información.

In [ ]:
coeficientes = {
    "OLS": ols.coef_,
    "Ridge": ridge.coef_,
    "Bayesiana": bayes.coef_,
    f"PCR (k={k_optimo})": beta_pcr_std,
}

fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True, sharey=True)
eje_x = np.arange(P)

for ax, (nombre, coef) in zip(axes.ravel(), coeficientes.items()):
    colores = np.where(np.isin(eje_x, INFORMATIVAS), "tab:red", "steelblue")
    ax.bar(eje_x, coef, color=colores, alpha=0.85)
    ax.scatter(eje_x, beta_true_std, color="black", s=18, zorder=3, label="β verdadero")
    ax.axhline(0, color="gray", linewidth=0.8)
    ax.set_title(nombre)
    ax.set_ylabel("peso")
    ax.legend(loc="upper right", fontsize=8)

for ax in axes[1]:
    ax.set_xlabel("variable")
fig.suptitle("Coeficientes estimados (escala estandarizada)")
fig.tight_layout()
plt.show()

In [ ]:
# Tabla resumen: magnitud de peso asignada a las variables informativas vs. el resto
filas = []
for nombre, coef in coeficientes.items():
    abs_coef = np.abs(coef)
    filas.append({
        "modelo": nombre,
        "|β| medio (informativas)": abs_coef[INFORMATIVAS].mean(),
        "|β| medio (resto)": np.delete(abs_coef, INFORMATIVAS).mean(),
    })
pd.DataFrame(filas).round(3)

### 4.7 Elección del modelo

**TODO:** elegir uno de los cuatro modelos y justificar (calidad de la recuperación de las variables informativas,
error de test/CV, interpretabilidad y robustez esperada frente al *shift*).

Punto a discutir: si $k$ quedó alto, el PCR se parece a OLS; si quedó bajo, actúa como filtro de ruido.
¿Qué pasó acá y qué implica para la comparación?

Redactar acá la decisión en un párrafo, por ejemplo:

> Se elige ... porque ...

In [ ]:
MODELO_ELEGIDO = "PCR"  # uno de: "OLS", "Ridge", "Bayesiana", "PCR"

modelos = {
    "OLS": ols,
    "Ridge": ridge,
    "Bayesiana": bayes,
    "PCR": pcr,
}
modelo = modelos[MODELO_ELEGIDO]
print("Modelo elegido:", MODELO_ELEGIDO)

## 5. PCA sobre el conjunto de entrenamiento

Se ajusta PCA sobre train (estandarizado) y se grafican las dos primeras componentes.

Es el **mismo ajuste** que usa el PCR: mismos datos, mismo escalado, solo cambia cuántas componentes se retienen.

In [ ]:
pca = PCA().fit(X_train_std)
var_ratio = pca.explained_variance_ratio_

print(f"PC1: {var_ratio[0]:.1%} de la varianza")
print(f"PC2: {var_ratio[1]:.1%} de la varianza")
print(f"PC1+PC2: {var_ratio[:2].sum():.1%}")
print(f"componentes para 90% de varianza: {np.searchsorted(np.cumsum(var_ratio), 0.90) + 1}")

In [ ]:
train_pca = pca.transform(X_train_std)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(np.arange(1, P + 1), np.cumsum(var_ratio), marker="o", markersize=3)
axes[0].axhline(0.90, color="gray", linestyle="--", linewidth=1)
axes[0].set_xlabel("número de componentes")
axes[0].set_ylabel("varianza acumulada")
axes[0].set_title("Scree plot")

sc = axes[1].scatter(
    train_pca[:, 0], train_pca[:, 1],
    c=y_train, cmap="viridis", s=30,
)
axes[1].set_xlabel(f"PC1 ({var_ratio[0]:.1%})")
axes[1].set_ylabel(f"PC2 ({var_ratio[1]:.1%})")
axes[1].set_title("Train proyectado (color = Y)")
fig.colorbar(sc, ax=axes[1], label="Y")

fig.tight_layout()
plt.show()

## 6. Construcción del conjunto test desplazado (A) y el original (B)

Se parte el test en dos mitades. Al conjunto A se le suma un vector aleatorio $\delta \sim U(-5, 5)^{50}$
a **todas las filas** (una constante por variable), lo que cambia la media pero no la covarianza.

In [ ]:
rng_shift = np.random.default_rng(SEED + 1)

indices = rng_shift.permutation(N_TEST)
indices_A, indices_B = indices[: N_TEST // 2], indices[N_TEST // 2 :]

A = X_test[indices_A].copy()
B = X_test[indices_B].copy()
y_A = y_test[indices_A]
y_B = y_test[indices_B]

delta = rng_shift.uniform(-5, 5, size=P)
A_shift = A + delta  # mismo delta en todas las filas

print(f"A: {A_shift.shape}  |  B: {B.shape}")
print(f"media de delta: {delta.mean():.2f}  |  rango: [{delta.min():.2f}, {delta.max():.2f}]")

## 7. Predicción sobre A y B

Se predice con el modelo elegido en el punto 4 (sin re-entrenar ni re-escalar) y se compara el error por observación
y el MSE de cada conjunto.

In [ ]:
y_pred_A = modelo.predict(scaler.transform(A_shift))
y_pred_B = modelo.predict(scaler.transform(B))

error_A = y_A - y_pred_A
error_B = y_B - y_pred_B


def resumen_errores(nombre, y_true, y_pred):
    e = y_true - y_pred
    return {
        "conjunto": nombre,
        "MSE": np.mean(e ** 2),
        "sesgo": e.mean(),
        "sesgo²": e.mean() ** 2,
        "varianza": e.var(),
    }


pd.DataFrame([
    resumen_errores("A (desplazado)", y_A, y_pred_A),
    resumen_errores("B (original)", y_B, y_pred_B),
]).round(3)

**TODO:** comentar los resultados. Guías para el análisis:

- ¿Cuánto más grande es el MSE en A que en B?
- ¿El error es principalmente **sesgo** (desplazamiento sistemático) o **varianza**?
- ¿Coincide el error extra con lo esperado a partir de $\beta \cdot \delta$? (para el modelo elegido, con $\delta$ conocido)

In [ ]:
# Chequeo teórico: como el shift mueve la media, el error esperado en A es -(β_efectivo · δ)
if MODELO_ELEGIDO == "PCR":
    beta_std_modelo = coefs_pcr_en_std(pcr)
else:
    beta_std_modelo = modelo.coef_
beta_efectivo = beta_std_modelo / scaler.scale_

sesgo_teorico = -(beta_efectivo @ delta)
print(f"sesgo teórico  -(β_ef · δ) : {sesgo_teorico:9.3f}")
print(f"sesgo observado en A       : {error_A.mean():9.3f}")
print(f"(β_ef · δ)²                : {sesgo_teorico ** 2:9.3f}")
print(f"MSE(A) - MSE(B)            : {np.mean(error_A ** 2) - np.mean(error_B ** 2):9.3f}")
print()
print(f"(β_true · δ)² (oráculo)    : {(beta_true @ delta) ** 2:.3f}")
print("La diferencia entre β_ef y β_true se amplifica porque δ es grande.")

## 8. Proyección de A y B en el espacio de componentes principales

Se usa el PCA ajustado en train (misma media y mismas componentes). Colorear por error cuadrático de predicción.

**Importante:** no re-ajustar ni re-centrar el PCA con A y B; si se centra cada conjunto por su propia media,
el desplazamiento desaparece del gráfico.

In [ ]:
pca_A = pca.transform(scaler.transform(A_shift))
pca_B = pca.transform(scaler.transform(B))

err2_A = error_A ** 2
err2_B = error_B ** 2
vmax = max(err2_A.max(), err2_B.max())

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)

for ax, (nombre, proy, err2) in zip(
    axes,
    [("A (desplazado)", pca_A, err2_A), ("B (original)", pca_B, err2_B)],
):
    ax.scatter(train_pca[:, 0], train_pca[:, 1], color="lightgray", s=12, label="train")
    sc = ax.scatter(proy[:, 0], proy[:, 1], c=err2, cmap="viridis", s=45, vmin=0, vmax=vmax, edgecolor="k", linewidth=0.3)
    ax.set_title(nombre)
    ax.set_xlabel(f"PC1 ({var_ratio[0]:.1%})")
    ax.legend(loc="best", fontsize=8)

axes[0].set_ylabel(f"PC2 ({var_ratio[1]:.1%})")
fig.colorbar(sc, ax=axes, label="error cuadrático")
plt.show()

**TODO:** comentar. Guías:

- ¿Dónde aparece A respecto de B y del train?
- ¿La dirección del desplazamiento coincide con alguna componente principal?
- ¿El error crece con la posición en el plano (por ejemplo, con PC1) o es uniforme?
- ¿Qué dice esto sobre la relación entre el error y la dirección del *shift*?

## 9. Estrategia si no se conocen las constantes

**TODO:** responder (en el informe) qué estrategia se elegiría para mejorar las predicciones en A sin conocer $\delta$.

Pistas para discutir:

1. **Detectar** el desplazamiento sin etiquetas (comparación de medias, test de dos muestras, clasificador A vs. train, PCA).
2. **Estimar** $\delta$ como diferencia de medias, ya que sumar una constante a todas las filas solo mueve la media.
3. **Corregir** restando el $\hat{\delta}$ estimado antes de predecir.
4. **Modelo robusto**: PCR con pocas componentes o Ridge, que dependen de menos direcciones del espacio.
5. Si hubiera etiquetas en A: recalibrar el intercepto o re-entrenar con datos aumentados.

La celda siguiente es una demostración opcional de la corrección por media.

In [ ]:
# Demostración: estimar el corrimiento como diferencia de medias (A vs. B, sin etiquetas) y corregir
delta_hat = A_shift.mean(axis=0) - B.mean(axis=0)
A_corregido = A_shift - delta_hat

y_pred_A_corr = modelo.predict(scaler.transform(A_corregido))
mse_A = np.mean(error_A ** 2)
mse_A_corr = np.mean((y_A - y_pred_A_corr) ** 2)

print(f"error de estimación de delta (norma): {np.linalg.norm(delta_hat - delta):.3f}")
print(f"MSE(A) sin corregir : {mse_A:.3f}")
print(f"MSE(A) corregido    : {mse_A_corr:.3f}")
print(f"MSE(B) referencia   : {np.mean(error_B ** 2):.3f}")

## 10. Conclusiones

**TODO:** sintetizar en pocas líneas:

- Qué modelo resultó mejor y por qué.
- Qué mostró el PCA sobre la estructura de los datos y sobre el desplazamiento.
- Cómo se relaciona el error con la posición en el espacio de componentes.
- Qué estrategia se propone frente a datos desplazados y qué limitaciones tiene.
